# 06.07 - PyTorch Tensors and a Tiny Classifier

**Daily output:** a tiny PyTorch classifier trained and evaluated correctly on toy data.

Today covers tensors, device, autograd, `nn.Module`, loss, optimizer, `model.train()`, `model.eval()`, and `torch.no_grad()`.

**Notebook type:** Solution notebook with full working code.


## Training Loop Contract

A PyTorch training step follows the same rhythm for toy data, CNNs, and transformers:

1. Move input and labels to the model device.
2. Run the forward pass.
3. Compute loss.
4. Clear old gradients with `optimizer.zero_grad()`.
5. Backpropagate with `loss.backward()`.
6. Update weights with `optimizer.step()`.

For `CrossEntropyLoss`, logits are `[batch, classes]` float values, and labels are `[batch]` `torch.long` class IDs.


In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


## Tensor Basics

Inspect `.shape`, `.dtype`, `.device`, and `.requires_grad`. Most PyTorch errors are shape, dtype, or device errors.


In [ ]:
x = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
y = torch.tensor([0, 1], dtype=torch.long)
print("x:", x.shape, x.dtype, x.device)
print("y:", y.shape, y.dtype, y.device)
print("moved:", x.to(device).device, y.to(device).device)


## Autograd

If a tensor has `requires_grad=True`, PyTorch records operations on it. Calling `.backward()` computes gradients for leaf tensors.


In [ ]:
w = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)
pred = w * 3 + b
loss = (pred - 10) ** 2
loss.backward()
print("loss:", float(loss))
print("w.grad:", w.grad.item())
print("b.grad:", b.grad.item())


## Toy 3-Class Data

This 2D dataset lets us train a classifier quickly while using the same code structure as a real image model.


In [ ]:
def make_blobs(n_per_class=120, noise=0.65, seed=42):
    g = torch.Generator().manual_seed(seed)
    centers = torch.tensor([[-2.0, -1.0], [2.0, -0.5], [0.0, 2.0]])
    xs, ys = [], []
    for class_id, center in enumerate(centers):
        xs.append(center + noise * torch.randn(n_per_class, 2, generator=g))
        ys.append(torch.full((n_per_class,), class_id, dtype=torch.long))
    X = torch.cat(xs)
    y = torch.cat(ys)
    perm = torch.randperm(len(y), generator=g)
    return X[perm], y[perm]

X, y = make_blobs()
train_n = int(0.8 * len(y))
train_ds = TensorDataset(X[:train_n], y[:train_n])
val_ds = TensorDataset(X[train_n:], y[train_n:])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)

xb, yb = next(iter(train_loader))
print("batch:", xb.shape, xb.dtype, yb.shape, yb.dtype)


## Model, Loss, Optimizer

`nn.Module` holds trainable layers. A linear classifier learns one score function per class. `CrossEntropyLoss` expects raw logits, not softmax probabilities.


In [ ]:
class TinyLinearClassifier(nn.Module):
    def __init__(self, in_features=2, num_classes=3):
        super().__init__()
        self.linear = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.linear(x)

model = TinyLinearClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)

logits = model(xb.to(device))
print(model)
print("logits:", logits.shape)
print("loss:", criterion(logits, yb.to(device)).item())


## Train and Evaluate Correctly

`model.train()` enables training behavior. `model.eval()` switches to inference behavior. Use `torch.no_grad()` during validation so PyTorch does not store a gradient graph.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_correct, total_seen = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(yb)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_seen += len(yb)
    return total_loss / total_seen, total_correct / total_seen

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_seen = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            preds = logits.argmax(1)
            total_loss += loss.item() * len(yb)
            total_correct += (preds == yb).sum().item()
            total_seen += len(yb)
            all_preds.append(preds.cpu())
            all_labels.append(yb.cpu())
    return {
        "loss": total_loss / total_seen,
        "accuracy": total_correct / total_seen,
        "preds": torch.cat(all_preds),
        "labels": torch.cat(all_labels),
    }


In [ ]:
for epoch in range(1, 31):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val = evaluate(model, val_loader, criterion, device)
    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:02d} | train {train_loss:.4f} acc {train_acc:.3f} | val {val['loss']:.4f} acc {val['accuracy']:.3f}")


## Macro-F1

Macro-F1 averages F1 across classes, so each class matters equally. This is more informative than accuracy when classes are imbalanced.


In [ ]:
def per_class_f1(preds, labels, num_classes):
    rows = []
    for c in range(num_classes):
        tp = int(((preds == c) & (labels == c)).sum())
        fp = int(((preds == c) & (labels != c)).sum())
        fn = int(((preds != c) & (labels == c)).sum())
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append({"class": c, "precision": precision, "recall": recall, "f1": f1, "support": int((labels == c).sum())})
    return rows

val = evaluate(model, val_loader, criterion, device)
rows = per_class_f1(val["preds"], val["labels"], num_classes=3)
for row in rows:
    print(row)
print("macro_f1:", sum(row["f1"] for row in rows) / len(rows))


## Common PyTorch Bugs

- Labels for `CrossEntropyLoss` must be `torch.long`.
- Logits for multi-class classification must be `[batch, classes]`.
- Model, inputs, and labels must be on the same device.
- Gradients accumulate unless you call `optimizer.zero_grad()`.
- Validation should use `model.eval()` and `torch.no_grad()`.


In [ ]:
# Debug examples: uncomment one at a time and read the error.
# criterion(model(xb.to(device)), yb.float().to(device))
# model.to(device)(xb)  # fails if device is cuda and xb stays on CPU


## Day 06 Checklist

Check input shape, label dtype, logits shape, device consistency, `model.train()` during training, `model.eval()` plus `torch.no_grad()` during validation, gradient clearing, and validation metrics computed on validation data.


## Test Cases

Run this cell after completing the TODO cells above. A correct implementation should print `Day 06 tests passed`.


In [ ]:
def run_day06_tests():
    required_names = [
        "make_blobs",
        "TinyLinearClassifier",
        "train_one_epoch",
        "evaluate",
        "per_class_f1",
    ]
    for name in required_names:
        assert name in globals(), f"Missing function or class: {name}"
        assert callable(globals()[name]), f"{name} must be callable"

    X_test, y_test = make_blobs(n_per_class=8, noise=0.5, seed=123)
    assert X_test.shape == (24, 2), f"Expected X shape (24, 2), got {X_test.shape}"
    assert y_test.shape == (24,), f"Expected y shape (24,), got {y_test.shape}"
    assert X_test.dtype == torch.float32
    assert y_test.dtype == torch.long
    assert set(y_test.tolist()) == {0, 1, 2}

    test_ds = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_ds, batch_size=12, shuffle=False)
    test_model = TinyLinearClassifier().to(device)
    test_criterion = nn.CrossEntropyLoss()
    test_optimizer = torch.optim.SGD(test_model.parameters(), lr=0.1)

    xb, yb = next(iter(test_loader))
    logits = test_model(xb.to(device))
    assert logits.shape == (12, 3), f"Expected logits shape (12, 3), got {logits.shape}"

    train_loss, train_acc = train_one_epoch(test_model, test_loader, test_criterion, test_optimizer, device)
    assert isinstance(train_loss, float)
    assert 0.0 <= train_acc <= 1.0

    metrics = evaluate(test_model, test_loader, test_criterion, device)
    for key in ["loss", "accuracy", "preds", "labels"]:
        assert key in metrics, f"evaluate output missing key: {key}"
    assert len(metrics["preds"]) == len(y_test)
    assert len(metrics["labels"]) == len(y_test)
    assert 0.0 <= metrics["accuracy"] <= 1.0

    perfect_rows = per_class_f1(
        preds=torch.tensor([0, 1, 2, 0, 1, 2]),
        labels=torch.tensor([0, 1, 2, 0, 1, 2]),
        num_classes=3,
    )
    assert len(perfect_rows) == 3
    assert all(abs(row["f1"] - 1.0) < 1e-8 for row in perfect_rows)

    print("Day 06 tests passed")

run_day06_tests()
